In [1]:
## Imports

import pandas as pd
import numpy as np
import datetime as dt

import yfinance as yf

from fredapi import Fred

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score
)

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# FRED API

#fred = Fred(api_key="YOUR_API_KEY_HERE")


# this for pulling the treasury yields: 2 and 10 year (could add more later)
# https://fred.stlouisfed.org/series/DGS2
# https://fred.stlouisfed.org/series/DGS10)


# Pull 2-year and 10-year Treasury yields from FRED
series_ids = {
    "DGS10": "10_Year_Treasury_Yield",
    "DGS2": "2_Year_Treasury_Yield"
}

fred_data = {}
for series, name in series_ids.items():
    fred_data[name] = fred.get_series(series)

fred_df = pd.DataFrame(fred_data)
fred_df.index = pd.to_datetime(fred_df.index)
fred_df.sort_index(inplace=True)

global_indices = [
    "^GSPC",
    "^DJI",
    "^NDX",
    "^HSI",
    "^N225",
    "000001.SS",
    "^STOXX50E",
    "^FTSE",
]

energy = ["XOM", "CVX", "SLB"]

sector_etfs = [
    "XLF", "XLK", "XLE", "XLY", "XLP",
    "XLV", "XLI", "XLB", "XLU", "IYR"
]

commodities = ["GLD", "SLV", "USO"]
usd = ["DX-Y.NYB"]
vol = ["^VIX"]

tickers = global_indices + energy + sector_etfs + commodities + vol + usd

start_date = "1995-01-01"
end_date = dt.datetime.now().strftime("%Y-%m-%d")

market_df = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    interval="1wk",
    auto_adjust=True,
    progress=False
)

adj_close = market_df["Close"].copy()

# Rename columns immediately so VIX survives downstream cleanly
adj_close = adj_close.rename(columns={
    "^VIX": "VIX",
    "DX-Y.NYB": "usd_index"
})

fred_weekly = fred_df.resample("W-FRI").last()
adj_close_weekly = adj_close.resample("W-FRI").last()

combine_sc = adj_close_weekly.join(fred_weekly, how="inner")
combine_sc = combine_sc.dropna(how="all", axis=1)
combine_sc = combine_sc.dropna(how="all", axis=0)

print(combine_sc.columns.tolist())
combine_sc.head()

['000001.SS', 'CVX', 'usd_index', 'GLD', 'IYR', 'SLB', 'SLV', 'USO', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY', 'XOM', '^DJI', '^FTSE', '^GSPC', '^HSI', '^N225', '^NDX', '^STOXX50E', 'VIX', '10_Year_Treasury_Yield', '2_Year_Treasury_Yield']


,000001.SS,CVX,usd_index,GLD,IYR,SLB,SLV,USO,XLB,XLE,...,^DJI,^FTSE,^GSPC,^HSI,^N225,^NDX,^STOXX50E,VIX,10_Year_Treasury_Yield,2_Year_Treasury_Yield
1995-01-06,NaN,7.065622,89.610001,NaN,NaN,6.349372,NaN,NaN,NaN,NaN,...,3867.409912,3065.000000,460.679993,7683.299805,19519.460938,401.589996,NaN,13.13,7.87,7.64
1995-01-13,NaN,7.045665,88.139999,NaN,NaN,6.241493,NaN,NaN,NaN,NaN,...,3908.459961,3048.300049,465.970001,7252.299805,19331.169922,410.480011,NaN,11.10,7.69,7.39
1995-01-20,NaN,7.384971,87.449997,NaN,NaN,6.549714,NaN,NaN,NaN,NaN,...,3869.429932,2995.000000,464.779999,7278.100098,18840.220703,410.130005,NaN,12.15,7.82,7.50
1995-01-27,NaN,7.384971,87.620003,NaN,NaN,6.673008,NaN,NaN,NaN,NaN,...,3857.989990,3022.199951,470.390015,7297.100098,18104.349609,407.220001,NaN,11.25,7.66,7.29
1995-02-03,NaN,7.305139,88.190002,NaN,NaN,6.642184,NaN,NaN,NaN,NaN,...,3928.639893,3059.699951,478.649994,7478.899902,18538.970703,416.140015,NaN,10.98,7.49,7.14


In [3]:
#Download historical market data
#Download macroeconomic data (Treasury yields) from FRED
#Convert all data to weekly frequency
#Merge market and macro data into one dataset
#Save prepared dataset for later steps

reg = ["SPY","QQQ","TLT","GLD","^VIX"]
etf = ["XLF","XLK","XLE","XLY","XLP","XLV","XLI","XLB","XLU","IYR"]
tickers = reg + etf

market_df = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    interval="1d",
    auto_adjust=True,
    progress=False
)

market_df = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    interval="1d",
    auto_adjust=True
)

close = market_df["Close"].copy()
close_weekly = close.resample("W-FRI").last()
fred_weekly = fred_df.resample("W-FRI").last().ffill()
combine_phuc = close_weekly.join(fred_weekly, how="inner")
combine_phuc = combine_phuc.rename(columns={"^VIX": "VIX"})
combine_phuc = combine_phuc.dropna(how="all", axis=1)
combine_phuc = combine_phuc.dropna(how="all", axis=0)
#print(list(combine_phuc.columns))
combine_phuc.head()
#combine_phuc.to_parquet("data_prep.parquet")


[*********************100%***********************]  15 of 15 completed


,GLD,IYR,QQQ,SPY,TLT,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,VIX,10_Year_Treasury_Yield,2_Year_Treasury_Yield
1995-01-06,NaN,NaN,NaN,26.665771,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.13,7.87,7.64
1995-01-13,NaN,NaN,NaN,27.063896,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.10,7.69,7.39
1995-01-20,NaN,NaN,NaN,26.955324,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.15,7.82,7.50
1995-01-27,NaN,NaN,NaN,27.281088,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.25,7.66,7.29
1995-02-03,NaN,NaN,NaN,27.814926,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.98,7.49,7.14


In [4]:
## remove overlapping before mapping to readable.

overlap = combine_sc.columns.intersection(combine_phuc.columns)

if len(overlap) > 0:
    print("Duplicate columns to review before merge:", list(overlap))
else:
    print("No duplicate columns found.")

Duplicate columns to review before merge: ['GLD', 'IYR', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY', 'VIX', '10_Year_Treasury_Yield', '2_Year_Treasury_Yield']


In [5]:
# Keep combine_sc as source of truth for overlapping columns
combine_phuc = combine_phuc.drop(columns=overlap, errors="ignore")

In [6]:
# Keep only dates where both data sources are available
merge_combine = combine_sc.merge(
    combine_phuc,
    left_index=True,
    right_index=True,
    how="inner"
)

In [7]:
ticker_map = {
    # Global indices
    "^GSPC": "SP500",
    "^DJI": "DOW_JONES",
    "^NDX": "NASDAQ100",
    "^HSI": "HANG_SENG",
    "^N225": "NIKKEI_225",
    "000001.SS": "SHANGHAI_COMPOSITE",
    "^STOXX50E": "EURO_STOXX_50",
    "^FTSE": "FTSE_100",

    # Market ETFs
    "SPY": "SP500_SPY",
    "QQQ": "NASDAQ100_QQQ",
    "TLT": "TREASURY_BOND_TLT",

    # Sector ETFs
    "XLF": "FINANCIALS_XLF",
    "XLK": "TECH_XLK",
    "XLE": "ENERGY_XLE",
    "XLY": "CONSUMER_DISCRETIONARY_XLY",
    "XLP": "CONSUMER_STAPLES_XLP",
    "XLV": "HEALTHCARE_XLV",
    "XLI": "INDUSTRIALS_XLI",
    "XLB": "MATERIALS_XLB",
    "XLU": "UTILITIES_XLU",
    "IYR": "REAL_ESTATE_IYR",

    # Commodities
    "GLD": "GOLD_GLD",
    "SLV": "SILVER_SLV",
    "USO": "CRUDE_OIL_USO",

    # Volatility index
    "^VIX": "VIX",

    # FRED data already readable but you can rename if you want
    "10_Year_Treasury_Yield": "TREASURY_10Y",
    "2_Year_Treasury_Yield": "TREASURY_2Y",

    "DX-Y.NYB" : "usd_index"
}

merge_combine = merge_combine.rename(columns=ticker_map)


In [8]:
# Drop duplicate VIX column after merge; keep the preferred source from combine_sc
#merge_combine = merge_combine.drop(columns=["VIX"], errors="ignore")

In [9]:
print("Final columns in merged dataset:")
print(list(merge_combine.columns))

Final columns in merged dataset:
['SHANGHAI_COMPOSITE', 'CVX', 'usd_index', 'GOLD_GLD', 'REAL_ESTATE_IYR', 'SLB', 'SILVER_SLV', 'CRUDE_OIL_USO', 'MATERIALS_XLB', 'ENERGY_XLE', 'FINANCIALS_XLF', 'INDUSTRIALS_XLI', 'TECH_XLK', 'CONSUMER_STAPLES_XLP', 'UTILITIES_XLU', 'HEALTHCARE_XLV', 'CONSUMER_DISCRETIONARY_XLY', 'XOM', 'DOW_JONES', 'FTSE_100', 'SP500', 'HANG_SENG', 'NIKKEI_225', 'NASDAQ100', 'EURO_STOXX_50', 'VIX', 'TREASURY_10Y', 'TREASURY_2Y', 'NASDAQ100_QQQ', 'SP500_SPY', 'TREASURY_BOND_TLT']


In [10]:
merge_combine.to_parquet("combined_weekly_dataset.parquet")

In [11]:
print("Merged dataset shape:", merge_combine.shape)
print("Date range:", merge_combine.index.min(), "to", merge_combine.index.max())
merge_combine.head()

Merged dataset shape: (1633, 31)
Date range: 1995-01-06 00:00:00 to 2026-04-17 00:00:00


,SHANGHAI_COMPOSITE,CVX,usd_index,GOLD_GLD,REAL_ESTATE_IYR,SLB,SILVER_SLV,CRUDE_OIL_USO,MATERIALS_XLB,ENERGY_XLE,...,HANG_SENG,NIKKEI_225,NASDAQ100,EURO_STOXX_50,VIX,TREASURY_10Y,TREASURY_2Y,NASDAQ100_QQQ,SP500_SPY,TREASURY_BOND_TLT
1995-01-06,NaN,7.065622,89.610001,NaN,NaN,6.349372,NaN,NaN,NaN,NaN,...,7683.299805,19519.460938,401.589996,NaN,13.13,7.87,7.64,NaN,26.665771,NaN
1995-01-13,NaN,7.045665,88.139999,NaN,NaN,6.241493,NaN,NaN,NaN,NaN,...,7252.299805,19331.169922,410.480011,NaN,11.10,7.69,7.39,NaN,27.063896,NaN
1995-01-20,NaN,7.384971,87.449997,NaN,NaN,6.549714,NaN,NaN,NaN,NaN,...,7278.100098,18840.220703,410.130005,NaN,12.15,7.82,7.50,NaN,26.955324,NaN
1995-01-27,NaN,7.384971,87.620003,NaN,NaN,6.673008,NaN,NaN,NaN,NaN,...,7297.100098,18104.349609,407.220001,NaN,11.25,7.66,7.29,NaN,27.281088,NaN
1995-02-03,NaN,7.305139,88.190002,NaN,NaN,6.642184,NaN,NaN,NaN,NaN,...,7478.899902,18538.970703,416.140015,NaN,10.98,7.49,7.14,NaN,27.814926,NaN


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=059a5e93-4459-411b-9a58-2a578fd7a892' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>